# ノートブック2：強化学習と動画書き出し（ランタイム：L4 または A100 GPU）【改訂版】

手順書の手順3・4・8をまとめたノートブックです。

**旧版からの主な変更**
- タスクを **`Unitree-G1-Tracking`（状態推定あり版）** に変更（シミュレーション専用なので、実機向けの制約を外して学習を楽にする）
- `train_sim.py` で外乱・摩擦のばらつきを緩める
- 学習ログはローカルに書き、**前面で動く監視セル**が 10 分ごとにドライブへ同期（バックグラウンド実行だけだとアイドル扱いで切断されやすい）
- 評価は**摩擦を固定**し、**転倒した時点で停止**（`render_policy.py` 改訂版）
- 予備ルート（同梱ダンス）は **131 秒の全編**を使う

## 3-1. ドライブの接続と共通設定

`NAME` で学習するお手本を選びます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

# ▼ 学習するお手本を選ぶ（1つだけ有効にする）
NAME = 'the_feels_full'       # サビ後半 20 秒（まずはお手本のデバッグ用に 3,000〜5,000 反復）
# NAME = 'the_feels_full'    # 全編 2分19秒（本番。20,000〜30,000 反復）
# NAME = 'lafan_dance_full'  # 予備ルート：同梱のモーションキャプチャ・ダンス 131 秒（手順8）
os.environ['NAME'] = NAME
os.environ['TASK'] = 'Unitree-G1-Tracking'   # 評価でも同じタスク名を使うこと
os.environ['START'] = str(SEG_START) if NAME == 'the_feels_20s' else '0'   # 比較動画用
L = '/content/unitree_rl_mjlab'

# ▼ 一括実行（「すべてのセルを実行」）用のスイッチ
RUN_SPEEDTEST = False   # 3-4 の速度測定。手動で測るときだけ True
RESUME = False          # 「切断されたときの再開」セル。再開するときだけ True にして、そのセルを手動で実行
os.environ['RUN_SPEEDTEST'] = '1' if RUN_SPEEDTEST else '0'
os.environ['RESUME'] = '1' if RESUME else '0'

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls $P/outputs/motions

## 3-2. unitree_rl_mjlab のインストール（5〜10分）

ドライブに学習ログがあれば（切断後の再開時）、ローカルに戻します。

In [ ]:
%%bash
set -e
pip -q install uv
bash $P/scripts/pin_clone.sh $URL_unitree_rl_mjlab /content/unitree_rl_mjlab $REV_unitree_rl_mjlab
uv venv /content/venv_mjlab --python 3.11 -q --clear
cd /content/unitree_rl_mjlab
# mjlab 1.2.0（2026年3月公開）は MuJoCo 系の版を固定していないため、そのままだと最新版が入り
# 「mujoco と mujoco-warp の版が合わない」エラーになる。公開当時の組み合わせに固定する。
cat > /content/mjlab_pins.txt <<'EOF'
mujoco==3.5.0
mujoco-warp==3.5.0.2
warp-lang==1.12.0
EOF
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt -e .     # mjlab==1.2.0 が入る
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt matplotlib scipy   # scipy は mjlab の地形モジュールが使う（依存に書かれていない）
# 学習・変換・評価で使う部品をここでまとめて読み込んで確認（足りなければこのセルで止まる）
/content/venv_mjlab/bin/python - <<'EOF'
import sys, os; sys.path.insert(0, os.getcwd())
import torch, mjlab, mujoco, mujoco_warp, warp, scipy, importlib.metadata as m
import mjlab.entity, mjlab.scene, mjlab.envs, mjlab.rl, mjlab.tasks, src.tasks
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| mjlab', m.version('mjlab'),
      '| mujoco', mujoco.__version__, '| mujoco-warp', m.version('mujoco-warp'), '| warp', warp.__version__)
print('[ok] 必要な部品はすべて読み込めました')
EOF
# 学習ログはローカルに置き、ドライブへは監視セルが同期する（ドライブにあれば戻す）
mkdir -p logs/rsl_rl $P/logs/rsl_rl
bash $P/scripts/sync_logs.sh $P/logs/rsl_rl logs/rsl_rl
ls logs/rsl_rl/*/ 2>/dev/null | tail -5 || true

## 3-3. お手本を学習用の形式（npz）に変換

30fps の CSV を 50fps に補間し、各リンクの位置・速度を計算して npz に保存します。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
if [ "$NAME" = "lafan_dance_full" ]; then
  cp src/assets/motions/g1/dance1_subject2.csv src/assets/motions/g1/$NAME.csv   # 予備ルート：同梱データ全編
else
  cp $P/outputs/motions/$NAME.csv src/assets/motions/g1/
fi
/content/venv_mjlab/bin/python scripts/csv_to_npz.py --robot g1 \
  --input-file src/assets/motions/g1/$NAME.csv --output-name $NAME \
  --input-fps 30 --output-fps 50
cp src/assets/motions/g1/$NAME.npz $P/outputs/motions/
ls -la src/assets/motions/g1/

## 3-4. 速度測定（50 反復だけ回す）

最後のほうの `Iteration time` を読み、本番の時間と CU を計算します（反復の速さはお手本の長さに関係しません）。

- 本番の時間［時間］＝ 反復回数 × `Iteration time` ÷ 3600
- 消費 CU ＝ 本番の時間 × GPU の CU/時（目安：L4 約5、A100 約13）

L4 が A100 の 2.5 倍より遅くなければ、L4 のほうが CU は安く済みます。両方で測ってから決めてください。

In [ ]:
%%bash
# 一括実行では速度測定を省略（3-1 で RUN_SPEEDTEST = True にしたときだけ実行）
if [ "$RUN_SPEEDTEST" != "1" ]; then echo "[skip] 速度測定は省略しました（RUN_SPEEDTEST = False）"; exit 0; fi
cd /content/unitree_rl_mjlab
/content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations 50 \
  --agent.logger tensorboard \
  --agent.run-name speedtest 2>&1 | grep -E "train_sim|Learning iteration|Iteration time|Mean episode length|ETA|Error|error" | tail -20
rm -rf logs/rsl_rl/g1_tracking/*_speedtest

## 3-5. 本番の学習

反復回数の目安：**20 秒版はお手本のデバッグなので 3,000〜5,000**、**全編（2分19秒）・同梱ダンス（131秒）は 20,000〜30,000**（リポジトリの既定値は 30,001）。
500 反復ごとに `model_XXX.pt` が保存されます。

学習はバックグラウンドで始め、**すぐ下の監視セルを実行したままにします**。監視セルが動いている間は Colab にアイドルとみなされにくく、10 分ごとにログとチェックポイントをドライブへ写します。
ブラウザのタブは開いたままにしてください（Pro はタブを閉じると切断されます）。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
ITERS=20000            # ← 20秒版 3000〜5000 / 全編・同梱ダンス 20000〜30000
# 学習がすでに動いていたら二重に起動しない
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then
  echo "[stop] 学習がすでに動いています（PID $(cat /content/train.pid)）。新しい学習は始めません。"; exit 1
fi
mkdir -p $P/logs
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations $ITERS \
  --agent.logger tensorboard \
  --agent.run-name $NAME \
  --push-scale 0.5 \
  > $P/logs/train_$NAME.log 2>&1 &
echo $! > /content/train.pid
echo $ITERS > /content/train.iters
echo "学習を開始しました（PID $(cat /content/train.pid)）。ログ: $P/logs/train_$NAME.log"

In [ ]:
# 監視セル：学習が終わるまで実行したままにする（止めても学習は止まらない）
import re, subprocess, time
from IPython.display import clear_output
pid = int(open('/content/train.pid').read())
log = f"{P}/logs/train_{NAME}.log"
alive = lambda: os.path.exists(f"/proc/{pid}")
sync = lambda: subprocess.run(['bash', f'{P}/scripts/sync_logs.sh', f'{L}/logs/rsl_rl', f'{P}/logs/rsl_rl'])
last = time.time()
while alive():
    time.sleep(60)
    txt = open(log, errors='ignore').read().splitlines()
    keep = [l.strip() for l in txt if re.search(r'Learning iteration|Mean episode length|Mean reward|ETA|Error|error', l)]
    clear_output(wait=True)
    print(time.strftime('%H:%M:%S'), f'学習中 (PID {pid})'); print('\n'.join(keep[-5:]))
    if time.time() - last > 600:
        sync(); last = time.time(); print('[sync] ドライブへ同期しました')
sync()
txt = open(log, errors='ignore').read()
print('学習プロセスが終了しました。ログの最後:'); print('\n'.join(txt.splitlines()[-15:]))

# 学習が正常に終わったかを確認（失敗していたら、ここで止めて後ろのセルを実行させない）
import glob
runs = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{NAME}*"), key=os.path.getmtime)
ckpts = glob.glob(f"{runs[-1]}/model_*.pt") if runs else []
its = [int(m) for m in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
want = int(open('/content/train.iters').read()) if os.path.exists('/content/train.iters') else None
print(f"[check] run: {runs[-1] if runs else 'なし'} / チェックポイント {len(ckpts)} 個 / 最後の反復 {max(its) if its else 'なし'} / 予定 {want}")
if 'Traceback' in txt or not ckpts:
    raise RuntimeError('学習が途中でエラー終了しました。上のログの最後を確認してください（後ろのセルは実行しません）。')
if want and its and max(its) < want * 0.95:
    raise RuntimeError(f'学習が予定の反復数まで進まずに終わりました（{max(its)}/{want}）。ログを確認してください。')
print('[ok] 学習は正常に終わりました。評価と動画の書き出しに進みます。')

In [ ]:
# 学習曲線（TensorBoard）。Episode length が 500（10秒）に近づくほど転ばなくなっている
%load_ext tensorboard
%tensorboard --logdir /content/unitree_rl_mjlab/logs/rsl_rl/g1_tracking

### 切断されたときの再開

**一括実行では自動でスキップされます。** 再開するときは、3-1 で `RESUME = True` にして 3-1・3-2・3-3 を実行し直してから（3-2 でドライブのログがローカルに戻ります）、次のセルを実行し、そのあと**監視セルをもう一度実行**します。
`ITERS` には「残りの反復回数」を入れます。

In [ ]:
%%bash
# 【切断されたときだけ実行】一括実行で意図しない追加学習が始まらないよう、RESUME = True のときだけ動く
if [ "$RESUME" != "1" ]; then echo "[skip] 再開セルは実行しません（3-1 で RESUME = False）"; exit 0; fi
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then
  echo "[stop] 学習がすでに動いています（PID $(cat /content/train.pid)）。再開しません。"; exit 1
fi
cd /content/unitree_rl_mjlab
ITERS=10000           # ← 残りの反復回数
RUN=$(basename $(ls -td logs/rsl_rl/g1_tracking/*_${NAME}* | head -1))
echo "再開する run: $RUN / 最新のチェックポイント: $(ls -v logs/rsl_rl/g1_tracking/$RUN/model_*.pt | tail -1)"
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations $ITERS \
  --agent.logger tensorboard \
  --agent.run-name ${NAME}_resume \
  --agent.resume True --agent.load-run "$RUN" \
  --push-scale 0.5 \
  > $P/logs/train_${NAME}_resume.log 2>&1 &
echo $! > /content/train.pid
echo $ITERS > /content/train.iters
echo started

## 3-6. 評価（学習中でも実行できます）

最新のチェックポイントで、摩擦 **0.8 で動画つき**、**0.5 と 1.0 で評価値だけ**を出します。3 つとも `success: true` なら合格です。
転倒した場合、動画はその時点で終わり、`first_fall_s` に時刻が記録されます。2-8 の物理チェックの「難所」と照らし合わせてください。

In [ ]:
%%bash
set -eo pipefail
cd /content/unitree_rl_mjlab
RUN=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}* | sed -n 1p)   # 一番新しい run（再開した run も含む）
[ -n "$RUN" ] || { echo "[stop] 学習の run が見つかりません"; exit 1; }
CKPT=$(ls -v $RUN/model_*.pt | tail -1)
TAG=$(basename $CKPT .pt)
[ -n "$CKPT" ] || { echo "[stop] チェックポイントが見つかりません（学習が終わっていない可能性）"; exit 1; }
echo "評価するチェックポイント: $CKPT"
mkdir -p $P/outputs/videos $P/outputs/eval
PY=/content/venv_mjlab/bin/python
$PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
  --checkpoint $CKPT --friction 0.8 --out $P/outputs/videos/policy_${NAME}_${TAG}.mp4 | tail -3
# 途中で転倒して動画が短い（または作られない）ときは、終了条件を外して全編の動画も書き出す（見せる用）
if python3 -c "import json,sys; sys.exit(0 if json.load(open('$P/outputs/videos/policy_${NAME}_${TAG}.json'))['success'] else 1)"; then
  echo "[ok] 転倒なしで最後まで踊れました"
else
  echo "[info] 転倒したため、終了条件を外した全編の動画も書き出します（_noterm.mp4）"
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction 0.8 --no-terminations --out $P/outputs/videos/policy_${NAME}_${TAG}_noterm.mp4 | grep -E "^\[" || true
fi
for F in 0.5 1.0; do
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction $F --no-video --out $P/outputs/eval/policy_${NAME}_${TAG}_mu$F.mp4 > /dev/null
done
python3 - <<EOF
import json, glob
fs = ["$P/outputs/videos/policy_${NAME}_${TAG}.json"] + sorted(glob.glob("$P/outputs/eval/policy_${NAME}_${TAG}_mu*.json"))
print(f"{'摩擦':>5} {'成功':>5} {'転倒時刻[s]':>10} {'関節誤差[rad]':>12} {'位置誤差[m]':>10}  止まった条件")
for f in fs:
    j = json.load(open(f))
    print(f"{j['friction']:>5} {str(j['success']):>5} {str(j['first_fall_s']):>10} {j['mean_joint_err_rad']:>12} {j['mean_body_pos_err_rel_m']:>10}  {j.get('fall_reason') or ''}")
import os
nt = "$P/outputs/videos/policy_${NAME}_${TAG}_noterm.json"
if os.path.exists(nt):
    j = json.load(open(nt))
    print(f"[終了条件なし・全編] 長さ {j['duration_s']} 秒 / 関節誤差 {j['mean_joint_err_rad']} rad / 位置誤差 平均 {j['mean_body_pos_err_rel_m']} m・最大 {j['max_body_pos_err_rel_m']} m")
EOF

In [ ]:
import glob
vids = sorted(glob.glob(f"{P}/outputs/videos/policy_{NAME}_model_*.mp4"), key=os.path.getmtime)
if not vids:
    print("動画がありません（評価で開始直後に止まった可能性）。3-6 の出力を確認してください。")
else:
    latest = vids[-1]
    print(latest, f"({os.path.getsize(latest) / 1e6:.1f} MB)")
    display(show(latest))

### （任意）学習の進み具合を表にする — 動画の「だんだん上手になる様子」とレポート用

数千反復おきのチェックポイントを、摩擦 0.8 で評価値だけ出します（1 つ 1〜2 分）。

In [ ]:
%%bash
cd /content/unitree_rl_mjlab
RUN=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}* | head -1)
mkdir -p $P/outputs/eval
for CKPT in $(ls -v $RUN/model_*.pt | awk 'NR%4==1'); do     # 2000 反復おき（500 ごとの保存の 4 つに 1 つ）
  TAG=$(basename $CKPT .pt)
  /content/venv_mjlab/bin/python $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction 0.8 --no-video --out $P/outputs/eval/progress_${NAME}_${TAG}.mp4 > /dev/null
  python3 -c "import json;j=json.load(open('$P/outputs/eval/progress_${NAME}_${TAG}.json'));print('$TAG', j['success'], j['first_fall_s'], j['completed_ratio'], j['mean_body_pos_err_rel_m'])"
done

## 4. 比較動画を作る

左：推定した人のメッシュ（全編の動画を `START` 秒からずらして使う）／中央：お手本／右：物理で踊る G1。音声は 30fps に変換した全編の動画から、同じく `START` 秒から付けます。

In [ ]:
%%bash
set -e
POLICY=$(ls -t $P/outputs/videos/policy_${NAME}_model_*.mp4 | head -1)   # 採用する動画に書き換えてよい
echo "採用: $POLICY  (START=$START)"
START=$START bash $P/scripts/make_comparison.sh \
  $P/outputs/gvhmr/the_feels_full/2_global.mp4 \
  $P/outputs/motions/${NAME}_retarget_preview.mp4 \
  "$POLICY" \
  $P/input/the_feels_full.mp4 \
  $P/outputs/videos/comparison_$NAME.mp4

In [ ]:
show(f"{P}/outputs/videos/comparison_{NAME}.mp4", width=1100)

## 8. 予備ルート：同梱のダンスデータ（131 秒）

3-1 で `NAME = 'lafan_dance_full'` にして、3-3 から実行します（3-3 が同梱 CSV を自動で使います）。
モーションキャプチャなので推定ノイズがなく、リポジトリの既定設定（30,001 反復）で学習される想定のデータです。
**The Feels の学習と並行して、最初に回しておく保険**です（手順書 7 のスケジュール参照）。
比較動画は人のメッシュがないため作れません。評価動画（3-6）をそのまま使います。